# Ingestion Pipeline Demo

This notebook demonstrates how the ingestion pipeline works:
fetching from sources, extracting candidates, classifying, deduplicating, and ranking.

In [ ]:
import sys
sys.path.insert(0, '..')

from ingestion.adapters.base import CandidateProblem
from extraction.classifiers.status_classifier import classify_rule_based, CandidateLabel
from extraction.classifiers.candidate_extractor import extract_from_text
from extraction.dedupe.deduplicator import find_duplicates
from extraction.dedupe.alias_linker import link_aliases
from extraction.ranking.scorer import rank_candidates

## Step 1: Extract candidates from text

Given a sample text that might appear in a survey paper, extract candidate problems.

In [ ]:
sample_text = """
Section 5: Open Problems

Conjecture 5.1 (Hadwiger, 1943): Every graph with no K_t minor is (t-1)-colorable.
This conjecture remains one of the deepest open problems in graph theory.

Open Problem 5.2: Is there a polynomial-time algorithm for graph isomorphism?
Babai (2016) showed a quasipolynomial-time algorithm, but the polynomial case is still open.

It remains open whether every bridgeless graph admits a nowhere-zero 5-flow.
This is a weakening of Tutte's 5-flow conjecture.

In future work, it would be interesting to explore connections between
spectral gap bounds and expansion properties of random graphs.
"""

candidates = extract_from_text(sample_text, source_id='demo_survey')
print(f'Extracted {len(candidates)} candidates:')
for c in candidates:
    print(f'  - {c.title}')
    print(f'    confidence: {c.confidence:.2f}')

## Step 2: Classify candidates

In [ ]:
print('Classification results:')
for c in candidates:
    result = classify_rule_based(f'{c.title} {c.statement}')
    print(f'  {c.title}')
    print(f'    label: {result.label.value}, confidence: {result.confidence:.2f}')
    print(f'    reason: {result.reason}')

## Step 3: Deduplicate against existing problems

In [ ]:
existing_titles = [
    'Hadwiger conjecture',
    'Graph Isomorphism Problem Complexity',
    'Cycle double cover conjecture',
]

unique, matches = find_duplicates(candidates, existing_titles, threshold=0.4)
print(f'Unique candidates: {len(unique)}')
print(f'Matched to existing: {len(matches)}')
for m in matches:
    print(f'  {m.candidate_a} ≈ {m.candidate_b} ({m.match_level}, sim={m.similarity:.2f})')

## Step 4: Rank remaining candidates

In [ ]:
ranked = rank_candidates(unique)
print('Ranked candidates:')
for s in ranked:
    print(f'  priority={s.priority_score:.2f}: {s.candidate.title}')
    print(f'    impact={s.impact_estimate:.2f} formality={s.formality_estimate:.2f} toolability={s.toolability_estimate:.2f}')

## Notes

This demo uses rule-based extraction and classification.
In production, the extraction step can be augmented with LLM calls
for higher-quality candidate identification, while the classifier
always allows `abstain` for uncertain cases.